# 01 — Getting Started with Hindsight

This notebook covers:
- Installing the Hindsight client
- Connecting to a Hindsight server
- Creating your first memory bank
- Basic retain, recall, and reflect operations
- Verifying your setup

**Prerequisites:** A running Hindsight server at `http://localhost:8888`

## 1. Setup and Installation

In [ ]:
# Install required packages (run once)
# !pip install hindsight-client nest_asyncio -U

In [2]:
# Import the client
from hindsight_client import Hindsight
import requests

# Connect to your Hindsight server
# Change the URL if your server is running elsewhere
HINDSIGHT_URL = "http://localhost:8888"

client = Hindsight(base_url=HINDSIGHT_URL)
print(f"Connected to Hindsight at {HINDSIGHT_URL}")

Connected to Hindsight at http://localhost:8888


In [ ]:
# Fix: Allow nested event loops in Jupyter
import nest_asyncio
nest_asyncio.apply()

## 2. Verify Your Setup

In [3]:
# Check health — the server and database should both be OK
resp = requests.get(f"{HINDSIGHT_URL}/health")
print(f"Health check: {resp.json()}")

# Check version — see what API version and features are enabled
resp = requests.get(f"{HINDSIGHT_URL}/version")
version_info = resp.json()
print(f"API version: {version_info.get('api_version', 'unknown')}")
print(f"Features: {version_info.get('features', {})}")

Health check: {'status': 'healthy', 'database': 'connected'}
API version: 0.8.3
Features: {'observations': True, 'mcp': True, 'worker': True, 'bank_config_api': True, 'bank_llm_health': False, 'file_upload_api': True, 'document_export_api': True, 'document_import_api': True, 'audit_log': False, 'llm_trace': True, 'store_document_text': True}


## 3. Create Your First Memory Bank

A **memory bank** is the container for all memories. Think of it as a database
for a specific agent, user, or project.

In [4]:
# Create a bank for our tutorial experiments
BANK_ID = "tutorial-getting-started"

client.create_bank(
    bank_id=BANK_ID,
    name="Getting Started Tutorial",
    reflect_mission="""
    I am a helpful tutorial assistant.
    I remember what users tell me about their preferences and projects.
    I prioritize practical, actionable advice.
    """
)
print(f"Created bank: {BANK_ID}")

RuntimeError: This event loop is already running

In [ ]:
# List all banks to confirm creation
banks = client.list_banks()
for bank in banks:
    print(f"  • {bank.id} — {bank.name} ({bank.fact_count} facts)")

## 4. Your First Memories: retain()

`retain()` stores information. Internally, Hindsight extracts entities,
relationships, and temporal data — then indexes everything for fast retrieval.

In [ ]:
# Store a simple fact
client.retain(
    bank_id=BANK_ID,
    content="Alice works at Google as a software engineer in Mountain View."
)
print("✓ Stored: Alice works at Google")

In [ ]:
# Store multiple facts with context and metadata
facts = [
    {"content": "Bob is a data scientist at Meta", "context": "career"},
    {"content": "Carol leads the ML team at Google", "context": "career"},
    {"content": "The team uses Python for all ML projects", "context": "tech-stack"},
    {"content": "Preferred cloud provider is AWS with ECS for deployments", "context": "infrastructure"},
]

for fact in facts:
    client.retain(
        bank_id=BANK_ID,
        content=fact["content"],
        context=fact.get("context")
    )
    print(f"✓ Stored: {fact['content'][:60]}...")

In [ ]:
# Store with a specific timestamp (when the event happened, not when stored)
from datetime import datetime, timezone

client.retain(
    bank_id=BANK_ID,
    content="Alice got promoted to Senior Engineer",
    timestamp=datetime(2026, 3, 15, tzinfo=timezone.utc).isoformat(),
    context="career-update"
)
print("✓ Stored with timestamp: Alice promoted March 2026")

## 5. Searching Memories: recall()

`recall()` uses TEMPR multi-strategy retrieval: Semantic + Keyword + Graph + Temporal.

In [ ]:
# Basic recall — search for anything about Alice
results = client.recall(
    bank_id=BANK_ID,
    query="What does Alice do?"
)

print(f"Found {len(results.results)} results:\n")
for r in results.results:
    print(f"  [{r.type}] {r.text}")

In [ ]:
# Search with type filtering — only "world" facts (objective facts)
results = client.recall(
    bank_id=BANK_ID,
    query="cloud infrastructure",
    types=["world"]
)

for r in results.results:
    print(f"  [{r.type}] {r.text}")

In [ ]:
# Search depth: "low" for quick lookups, "high" for thorough searches
quick = client.recall(bank_id=BANK_ID, query="Alice", budget="low")
deep = client.recall(bank_id=BANK_ID, query="Alice", budget="high")

print(f"Low budget: {len(quick.results)} results")
print(f"High budget: {len(deep.results)} results")

## 6. Reasoning Over Memories: reflect()

`reflect()` doesn't just retrieve — it synthesizes an answer from the memories,
applying the bank's mission and disposition.

In [ ]:
# Ask Hindsight to reason about our stored knowledge
answer = client.reflect(
    bank_id=BANK_ID,
    query="Summarize what we know about the team and their tech stack.",
    budget="high"
)

print("=== Generated Answer ===")
print(answer.text)
print(f"\n---")
print(f"Based on {len(answer.based_on)} source memories")
print(f"Token usage: {answer.usage.total_tokens} total tokens")

In [ ]:
# Show evidence — trace every claim to its source
print("Evidence for the answer above:\n")
for i, source in enumerate(answer.based_on, 1):
    print(f"  {i}. [{source.type}] {source.text[:120]}...")

## 7. Bank Statistics

Check what's in your bank.

In [ ]:
# Get bank statistics
stats = client.get_bank_stats(bank_id=BANK_ID)

print("=== Bank Statistics ===")
print(f"Total memories: {stats.get('total_memories', 'N/A')}")
print(f"World facts: {stats.get('world_facts', 'N/A')}")
print(f"Experience facts: {stats.get('experience_facts', 'N/A')}")
print(f"Observations: {stats.get('observations', 'N/A')}")
print(f"Pending operations: {stats.get('pending_operations', 'N/A')}")

## 8. Cleanup

Remove the tutorial bank to keep things tidy.

In [ ]:
# Uncomment to delete the tutorial bank
# client.delete_bank(bank_id=BANK_ID)
# print(f"Deleted bank: {BANK_ID}")

# Or keep it for the next notebook
print("Bank preserved for future notebooks.")

## Summary

In this notebook you:
1. Connected to a Hindsight server
2. Created a memory bank with a custom mission
3. Stored facts with `retain()` — with context, timestamps, and metadata
4. Searched with `recall()` — multi-strategy retrieval
5. Reasoned with `reflect()` — synthesized answer from memories
6. Checked bank statistics

**Next:** [03_retain.ipynb](03_retain.ipynb) — dive deep into retain